# SCOPE $w_p$ convergence

Demonstrates that the **SCOPE sub-volume correction** recovers the full-box projected
two-point correlation function $w_p(r_p)$ from a fraction of the simulation sub-volumes.

**Three analyses:**

1. **Quick-look** — 4-panel convergence figure for a single config/redshift.
2. **Headline comparison** — naïve vs SCOPE on the same panel, showing the correction impact.
3. **Convergence threshold** — median $|\Delta w_p/w_p|$ vs $n_{\rm subvol}$: the "how many sub-volumes do you need?" plot.
4. **Full sweep** — all redshifts × mass cuts.

**Data:** `scripts/2pcf/slurm/submit_scope_wp_split_by_n.sh`  
**Redshifts:** iz155 ($z\approx1.5$), iz207 ($z\approx0.5$), iz271 ($z=0$)

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../src').resolve()))
from utils.matplotlib_config import setconfig
from config import get_snapshot_redshift

setconfig()

In [ ]:
DATA_ROOT     = Path('../../data/2pcf/scope_wp')
DATA_ROOT_LRP = Path('../../data/2pcf/scope_wp_large_rp')
MODEL     = 'lc16'
CENTRALS  = 1
N_REF     = 1024   # reference n_subvol (full box)

# ── Quick-look config (sections 1–2) ──────────────────────────────────────────
QK_IZ        = 155
QK_MSTAR_TAG = 'mstar10.0'
QK_MHALO_TAG = 'none'
QK_Z         = get_snapshot_redshift(f'iz{QK_IZ}')

# ── n values to show in detail plots ──────────────────────────────────────────
PLOT_N       = [4, 16, 64, 256, 512]
HEADLINE_N   = [4, 16, 64, 256]

print(f'Quick-look: iz{QK_IZ}  →  z = {QK_Z:.3f}')

## Helper functions

In [ ]:
def load_and_aggregate(iz, mstar_tag, mhalo_tag, data_root=None):
    """Load all CSVs for one (iz, mstar, mhalo) config and aggregate over seeds.

    Reference (n=1024) is loaded exclusively from seed*/n1024/ paths so it always
    comes from the dedicated split-by-n full-box runs, not from multi-n files where
    n_total is estimated differently.  All other n values are loaded from the
    multi-n files and/or seed*/n{N}/ paths, with n=1024 rows stripped out.
    """
    if data_root is None:
        data_root = DATA_ROOT
    rdir = data_root / MODEL / f'iz{iz}' / f'centrals_{CENTRALS}' / mstar_tag / mhalo_tag

    # ── Non-reference data ────────────────────────────────────────────────────
    data_files = (sorted(rdir.glob(f'seed*/scope_wp_*_iz{iz}.csv')) +
                  [f for f in sorted(rdir.glob(f'seed*/n*/scope_wp_*_iz{iz}.csv'))
                   if f'/n{N_REF}/' not in str(f)])

    # ── Reference (n=1024): explicit split-by-n path only ─────────────────────
    ref_files = sorted(rdir.glob(f'seed*/n{N_REF}/scope_wp_*_iz{iz}.csv'))

    if not data_files and not ref_files:
        return None

    dfs = []
    if data_files:
        df_data = pd.concat([pd.read_csv(f) for f in data_files], ignore_index=True)
        df_data = df_data[df_data['n_subvol'] != N_REF]          # strip any n=1024 from multi-n files
        df_data = df_data.drop_duplicates(subset=['selection_seed', 'n_subvol', 'bin_idx'])
        dfs.append(df_data)

    if ref_files:
        df_ref = pd.concat([pd.read_csv(f) for f in ref_files], ignore_index=True)
        df_ref = df_ref[df_ref['n_subvol'] == N_REF]
        df_ref = df_ref.drop_duplicates(subset=['bin_idx'])       # one per bin; seed is irrelevant
        dfs.append(df_ref)
    else:
        print(f'  WARNING: n={N_REF} reference not yet available for iz{iz} '
              f'{mstar_tag}/{mhalo_tag}')

    df = pd.concat(dfs, ignore_index=True)

    g = df.groupby(['n_subvol', 'bin_idx', 'r_p'], as_index=False).agg(
        wp_corr_mean =('wp_corrected', 'mean'),
        wp_corr_std  =('wp_corrected', 'std'),
        wp_naive_mean=('wp_naive',     'mean'),
        wp_naive_std =('wp_naive',     'std'),
        n_seeds      =('wp_corrected', 'count'),
    )
    g['wp_corr_err']  = g['wp_corr_std']  / np.sqrt(g['n_seeds'])
    g['wp_naive_err'] = g['wp_naive_std'] / np.sqrt(g['n_seeds'])

    ref_c = (g[g['n_subvol'] == N_REF][['r_p', 'wp_corr_mean']]
               .rename(columns={'wp_corr_mean': 'wp_ref'}))
    ref_n = (g[g['n_subvol'] == N_REF][['r_p', 'wp_naive_mean']]
               .rename(columns={'wp_naive_mean': 'wp_naive_ref'}))
    g = g.merge(ref_c, on='r_p', how='left').merge(ref_n, on='r_p', how='left')

    # log₁₀ residuals — NaN where either side is non-positive
    pos_c = (g['wp_corr_mean'] > 0) & (g['wp_ref'] > 0)
    pos_n = (g['wp_naive_mean'] > 0) & (g['wp_naive_ref'] > 0)
    g['dlog_diff_corr']  = np.where(pos_c, np.log10(g['wp_corr_mean']  / g['wp_ref']),       np.nan)
    g['dlog_diff_naive'] = np.where(pos_n, np.log10(g['wp_naive_mean'] / g['wp_naive_ref']), np.nan)
    # propagated log₁₀ errors: d(log₁₀ x) ≈ dx / (x ln 10)
    g['dlog_err_corr']  = np.where(pos_c, g['wp_corr_err']  / (g['wp_corr_mean'].abs()  * np.log(10)), np.nan)
    g['dlog_err_naive'] = np.where(pos_n, g['wp_naive_err'] / (g['wp_naive_mean'].abs() * np.log(10)), np.nan)
    return g


def _n_colors(n_values):
    """Rainbow color progression: blue (small n) → red (large n)."""
    cmap = mpl.colormaps['rainbow']
    return [cmap(i / max(len(n_values) - 1, 1)) for i in range(len(n_values))]


def _plot_naive_scope_columns(agg_df, pn, fig, axes):
    """Fill a 2×2 axes array: naive (left) and SCOPE (right) columns.

    Top row: w_p.  Bottom row: Δlog₁₀ w_p vs full-box reference.
    """
    ax_tl, ax_tr = axes[0]
    ax_bl, ax_br = axes[1]
    colors = _n_colors(pn)

    ref = agg_df[agg_df['n_subvol'] == N_REF].sort_values('r_p')
    if not ref.empty:
        ax_tl.plot(ref['r_p'], ref['wp_naive_mean'], color='black', lw=2.5, zorder=10,
                   label=f'$N_{{\\rm subvol}}={N_REF}$ (full box)')
        ax_tr.plot(ref['r_p'], ref['wp_corr_mean'],  color='black', lw=2.5, zorder=10,
                   label=f'$N_{{\\rm subvol}}={N_REF}$ (full box)')
    ax_bl.axhline(0, color='black', lw=1.5, ls='--', zorder=10)
    ax_br.axhline(0, color='black', lw=1.5, ls='--', zorder=10)

    for n, col in zip(pn, colors):
        sub = agg_df[agg_df['n_subvol'] == n].sort_values('r_p')
        pos_n = sub['wp_naive_mean'] > 0
        pos_c = sub['wp_corr_mean']  > 0
        kw = dict(fmt='s', color=col, ms=5, lw=1.2, elinewidth=0.8, capsize=2,
                  label=f'$N_{{\\rm subvol}}={n}$')
        ax_tl.errorbar(sub.loc[pos_n, 'r_p'], sub.loc[pos_n, 'wp_naive_mean'],
                       yerr=sub.loc[pos_n, 'wp_naive_err'], **kw)
        ax_tr.errorbar(sub.loc[pos_c, 'r_p'], sub.loc[pos_c, 'wp_corr_mean'],
                       yerr=sub.loc[pos_c, 'wp_corr_err'], **kw)
        kw_bot = dict(fmt='s', color=col, ms=5, lw=1.2, elinewidth=0.8, capsize=2)
        valid_n = sub['dlog_diff_naive'].notna()
        valid_c = sub['dlog_diff_corr'].notna()
        ax_bl.errorbar(sub.loc[valid_n, 'r_p'], sub.loc[valid_n, 'dlog_diff_naive'],
                       yerr=sub.loc[valid_n, 'dlog_err_naive'], **kw_bot)
        ax_br.errorbar(sub.loc[valid_c, 'r_p'], sub.loc[valid_c, 'dlog_diff_corr'],
                       yerr=sub.loc[valid_c, 'dlog_err_corr'],  **kw_bot)

    for ax in [ax_tl, ax_tr]:
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_ylabel(r'$w_p(r_p)$ [Mpc/$h$]')
        ax.legend(ncol=2, fontsize=9)
    for ax in [ax_bl, ax_br]:
        ax.set_xscale('log')
        ax.set_xlabel(r'$r_p$ [$h^{-1}$Mpc]')
        ax.set_ylabel(r'$\Delta\log_{10} w_p$')

    ax_tl.set_title('Naïve (no correction)')
    ax_tr.set_title('SCOPE corrected')
    ax_bl.set_title(r'Naïve — $\Delta\log_{10} w_p$ vs full box')
    ax_br.set_title(r'SCOPE — $\Delta\log_{10} w_p$ vs full box')

    handles = [mpl.lines.Line2D([0],[0], color=c, lw=2, marker='s', ms=5,
                                label=f'$N_{{\\rm subvol}}={n}$')
               for n, c in zip(pn, colors)]
    ax_bl.legend(handles=handles, ncol=2, fontsize=9)
    ax_br.legend(handles=handles, ncol=2, fontsize=9)


def four_panel(agg_df, iz, z, mstar_tag, mhalo_tag, plot_n_sel):
    """4-panel: (naïve, SCOPE) × (w_p, Δlog₁₀ w_p residual)."""
    avail = sorted(agg_df['n_subvol'].unique())
    pn    = [n for n in plot_n_sel if n in avail]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    _plot_naive_scope_columns(agg_df, pn, fig, axes)
    fig.suptitle(
        f'$w_p$ convergence — L800/{MODEL}  iz{iz} ($z={z:.2f}$)'
        f'  ({mstar_tag} / mhalo={mhalo_tag})',
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


def headline_comparison(iz, z, mstar_tag, mhalo_tag, n_show=None):
    """Naïve (left) vs SCOPE (right) on separate columns for selected n values."""
    g = load_and_aggregate(iz, mstar_tag, mhalo_tag)
    if g is None:
        print(f'No data: iz{iz} {mstar_tag}/{mhalo_tag}')
        return
    avail  = sorted(g['n_subvol'].unique())
    n_show = [n for n in (n_show or [4, 16, 64, 256]) if n in avail]
    if not n_show:
        print(f'No requested n values available for iz{iz} {mstar_tag}/{mhalo_tag}')
        return
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    _plot_naive_scope_columns(g, n_show, fig, axes)
    fig.suptitle(
        f'Naïve vs SCOPE — L800/{MODEL}  iz{iz} ($z={z:.2f}$)'
        f'  ({mstar_tag} / mhalo={mhalo_tag})',
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


def convergence_threshold(iz_list, configs):
    """Plot median |Δlog₁₀ w_p| vs n_subvol — the 'how many sub-volumes?' plot."""
    cfg_colors = mpl.colormaps['tab10'](np.linspace(0, 0.7, max(len(configs), 1)))
    iz_markers = {iz: m for iz, m in zip(iz_list, ['o', 's', '^', 'D'])}

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    ax_l.set_title('SCOPE corrected')
    ax_r.set_title('Naïve (no correction)')

    for thresh in [0.01, 0.05, 0.1]:
        for ax in [ax_l, ax_r]:
            ax.axhline(thresh, color='lightgrey', lw=1.0, ls=':', zorder=0)
            ax.text(1.01, thresh, f'{thresh:.2f}', transform=ax.get_yaxis_transform(),
                    fontsize=8, va='center', color='grey')

    for (mstar_tag, mhalo_tag), cfg_col in zip(configs, cfg_colors):
        for iz in iz_list:
            z = get_snapshot_redshift(f'iz{iz}')
            g = load_and_aggregate(iz, mstar_tag, mhalo_tag)
            if g is None:
                continue
            ns = sorted(n for n in g['n_subvol'].unique() if n != N_REF)
            if not ns:
                continue
            med_c = [np.nanmedian(np.abs(g[g['n_subvol']==n]['dlog_diff_corr'].dropna()))  for n in ns]
            med_n = [np.nanmedian(np.abs(g[g['n_subvol']==n]['dlog_diff_naive'].dropna())) for n in ns]
            label = f'{mstar_tag}/{mhalo_tag}  $z={z:.1f}$'
            mkr   = iz_markers.get(iz, 'o')
            ax_l.plot(ns, med_c, marker=mkr, color=cfg_col, lw=1.8, ms=6, label=label)
            ax_r.plot(ns, med_n, marker=mkr, color=cfg_col, lw=1.8, ms=6, label=label)

    for ax in [ax_l, ax_r]:
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel(r'$N_{\rm subvol}$')
        ax.set_ylabel(r'median $|\Delta\log_{10} w_p|$')
        ax.legend(fontsize=8, ncol=1, loc='upper right')

    fig.suptitle(f'Convergence threshold — L800/{MODEL}', fontsize=13)
    plt.tight_layout()
    plt.show()


print('Helper functions defined.')

## 1  Quick-look (single config)

In [ ]:
g = load_and_aggregate(QK_IZ, QK_MSTAR_TAG, QK_MHALO_TAG)
if g is None:
    print(f'No data yet for iz{QK_IZ} {QK_MSTAR_TAG}/{QK_MHALO_TAG}')
else:
    four_panel(g, QK_IZ, QK_Z, QK_MSTAR_TAG, QK_MHALO_TAG, PLOT_N)

## 2  Headline: naïve vs SCOPE on the same panel

In [ ]:
headline_comparison(QK_IZ, QK_Z, QK_MSTAR_TAG, QK_MHALO_TAG, n_show=HEADLINE_N)

## 3  Convergence threshold

How many sub-volumes are needed for <5% accuracy?
Left panel shows SCOPE recovers the full box; right shows naïve diverges badly.

In [ ]:
THRESHOLD_IZ_LIST = [155, 207, 271]
THRESHOLD_CONFIGS = [
    ('mstarnone', 'none'),
    ('mstarnone', '1e9'),
    ('mstar10.0', 'none'),
    ('mstar11.0', 'none'),
]
convergence_threshold(THRESHOLD_IZ_LIST, THRESHOLD_CONFIGS)

## 4  Full sweep — all redshifts × mass cuts

In [ ]:
IZ_LIST = [155, 207, 271]
CONFIGS = [
    ('mstarnone', 'none'),
    ('mstarnone', '1e9'),
    ('mstar10.0', 'none'),
    ('mstar11.0', 'none'),
]

for iz in IZ_LIST:
    z = get_snapshot_redshift(f'iz{iz}')
    print(f'\n{"="*60}\niz{iz}  z = {z:.3f}\n{"="*60}')
    for mstar_tag, mhalo_tag in CONFIGS:
        g = load_and_aggregate(iz, mstar_tag, mhalo_tag)
        if g is None:
            print(f'  No data yet: {mstar_tag}/{mhalo_tag} — skipping')
            continue
        n_avail = sorted(g['n_subvol'].unique())
        print(f'  {mstar_tag}/{mhalo_tag} — n_subvol: {n_avail}')
        four_panel(g, iz, z, mstar_tag, mhalo_tag, PLOT_N)
        headline_comparison(iz, z, mstar_tag, mhalo_tag, n_show=HEADLINE_N)

## 5  Large-$r_p$ extension check

Temporary diagnostic: loads `scope_wp` (small-scale, $r_p < 31.6\,h^{-1}$Mpc) and
`scope_wp_large_rp` ($r_p > 31.6\,h^{-1}$Mpc) side by side on the same axes.
The dotted vertical line marks the join.  Once the large-rp jobs complete and the
datasets look consistent, combine the CSVs and remove this section.

In [ ]:
def plot_combined_rp(iz, mstar_tag, mhalo_tag, n_show=None):
    """Plot SCOPE w_p combining small-scale and large-rp campaigns on the same axes.

    Top panel: w_p(r_p).  Bottom panel: Δlog₁₀ w_p vs n=1024 reference.
    Dotted vertical line marks the r_p join at 31.6 Mpc/h.
    Each dataset uses its own n=1024 as the reference for the residual panel.
    """
    z   = get_snapshot_redshift(f'iz{iz}')
    g_s = load_and_aggregate(iz, mstar_tag, mhalo_tag)
    g_l = load_and_aggregate(iz, mstar_tag, mhalo_tag, data_root=DATA_ROOT_LRP)

    if g_s is None and g_l is None:
        print(f'No data for iz{iz} {mstar_tag}/{mhalo_tag}')
        return

    all_n = set()
    for g in [g_s, g_l]:
        if g is not None:
            all_n.update(g['n_subvol'].unique())

    n_show = n_show or [4, 16, 64, 256, 512, N_REF]
    n_plot = [n for n in n_show if n in all_n and n != N_REF]
    colors = _n_colors(n_plot)
    cmap   = dict(zip(n_plot, colors))

    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

    seen_n      = set()
    ref_plotted = False
    for g in [g_s, g_l]:
        if g is None:
            continue
        ref = g[g['n_subvol'] == N_REF].sort_values('r_p')
        if not ref.empty:
            lbl = f'$N_{{\\rm subvol}}={N_REF}$ (full box)' if not ref_plotted else '_nolegend_'
            ax_top.plot(ref['r_p'], ref['wp_corr_mean'], color='black', lw=2.5,
                        zorder=10, label=lbl)
            ref_plotted = True

        for n in n_plot:
            sub = g[g['n_subvol'] == n].sort_values('r_p')
            if sub.empty:
                continue
            col = cmap[n]
            lbl = f'$N_{{\\rm subvol}}={n}$' if n not in seen_n else '_nolegend_'
            pos = sub['wp_corr_mean'] > 0
            ax_top.errorbar(sub.loc[pos, 'r_p'], sub.loc[pos, 'wp_corr_mean'],
                            yerr=sub.loc[pos, 'wp_corr_err'],
                            fmt='s', color=col, ms=5, lw=1.2, elinewidth=0.8,
                            capsize=2, label=lbl)
            valid = sub['dlog_diff_corr'].notna()
            ax_bot.errorbar(sub.loc[valid, 'r_p'], sub.loc[valid, 'dlog_diff_corr'],
                            yerr=sub.loc[valid, 'dlog_err_corr'],
                            fmt='s', color=col, ms=5, lw=1.2, elinewidth=0.8, capsize=2)
            seen_n.add(n)

    ax_bot.axhline(0, color='black', lw=1.5, ls='--', zorder=10)

    for ax in [ax_top, ax_bot]:
        ax.axvline(31.6227766017, color='grey', lw=1, ls=':', alpha=0.7)

    ax_top.set_xscale('log'); ax_top.set_yscale('log')
    ax_top.set_ylabel(r'$w_p(r_p)$ [Mpc/$h$]')
    ax_top.legend(ncol=2, fontsize=9)
    ax_top.set_title(
        f'SCOPE $w_p$ — combined $r_p$ range\n'
        f'L800/{MODEL}  iz{iz} ($z={z:.2f}$)  {mstar_tag}/{mhalo_tag}'
    )

    ax_bot.set_xscale('log')
    ax_bot.set_xlabel(r'$r_p$ [$h^{-1}$Mpc]')
    ax_bot.set_ylabel(r'$\Delta\log_{10} w_p$')
    ax_bot.set_title(r'Fractional residual vs full box — $\Delta\log_{10} w_p$ (SCOPE corrected)')
    handles = [mpl.lines.Line2D([0],[0], color=c, lw=2, marker='s', ms=5,
                                label=f'$N_{{\\rm subvol}}={n}$')
               for n, c in zip(n_plot, colors)]
    ax_bot.legend(handles=handles, ncol=2, fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Demo: run for the two configs we submitted large-rp jobs for ───────────────
COMBINED_IZ_LIST = [155, 207, 271]
COMBINED_CONFIGS = [
    ('mstarnone', 'none'),
    ('mstarnone', '1e9'),
]

for iz in COMBINED_IZ_LIST:
    for mstar_tag, mhalo_tag in COMBINED_CONFIGS:
        plot_combined_rp(iz, mstar_tag, mhalo_tag, n_show=[4, 16, 64, 256, 512, N_REF])

In [ ]:
run_dir   = DATA_ROOT / MODEL / f'iz{IZ}' / f'centrals_{CENTRALS}' / MSTAR_TAG / MHALO_TAG
csv_files = sorted(run_dir.glob(f'seed*/scope_wp_*_iz{IZ}.csv'))

if not csv_files:
    raise FileNotFoundError(
        f'No CSVs under {run_dir}.\n'
        'Run:  bash scripts/2pcf/slurm/submit_scope_wp_l800_lc16_campaign.sh'
    )

raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

n_values = sorted(raw['n_subvol'].unique())
print(f'{len(csv_files)} files | {raw["selection_seed"].nunique()} seeds | n_subvol: {n_values}')

# Mean over seeds for both corrected and naive
agg = (
    raw.groupby(['n_subvol', 'bin_idx', 'r_p'], as_index=False)
    .agg(
        wp_corr_mean  = ('wp_corrected', 'mean'),
        wp_naive_mean = ('wp_naive',     'mean'),
    )
)

# Reference log10 w_p from n=N_REF (corrected)
ref_corr  = (
    agg[agg['n_subvol'] == N_REF][['r_p', 'wp_corr_mean']]
    .rename(columns={'wp_corr_mean': 'wp_corr_ref'})
)
ref_naive = (
    agg[agg['n_subvol'] == N_REF][['r_p', 'wp_naive_mean']]
    .rename(columns={'wp_naive_mean': 'wp_naive_ref'})
)

agg = agg.merge(ref_corr,  on='r_p', how='left')
agg = agg.merge(ref_naive, on='r_p', how='left')

EPS = 1e-30
agg['log10_wp_corr']  = np.log10(agg['wp_corr_mean'].clip(lower=EPS))
agg['log10_wp_naive'] = np.log10(agg['wp_naive_mean'].clip(lower=EPS))

agg['dlog10_wp_corr']  = agg['log10_wp_corr']  - np.log10(agg['wp_corr_ref'].clip(lower=EPS))
agg['dlog10_wp_naive'] = agg['log10_wp_naive'] - np.log10(agg['wp_naive_ref'].clip(lower=EPS))

agg['log10_r_p'] = np.log10(agg['r_p'])

print(agg[['n_subvol', 'r_p', 'log10_wp_corr', 'dlog10_wp_corr']].head(8).to_string(index=False))


cmap   = mpl.colormaps['rainbow']
colors = cmap(np.linspace(0.0, 1.0, len(n_values)))
marker = 's'   # square markers, matching Image 2
ms     = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
ax_tl, ax_tr, ax_bl, ax_br = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

def plot_panel(ax, col, ref_col, ylabel, title, is_residual=False):
    for n, color in zip(n_values, colors):
        sub = agg[agg['n_subvol'] == n].sort_values('log10_r_p')
        lw  = 2.5 if n == N_REF else 1.6
        alpha = 1.0 if n == N_REF else 0.85
        ax.plot(
            sub['log10_r_p'], sub[col],
            marker=marker, ms=ms, lw=lw, alpha=alpha,
            color=color, label=f'$N_{{\\rm subvol}}={n}$',
        )
    if is_residual:
        ax.axhline(0, color='black', lw=1.5, ls='--', zorder=5)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)

plot_panel(ax_tl, 'log10_wp_naive', 'wp_naive_ref',
           r'$\log_{10}\,w_p$', 'Naïve (no correction)')
ax_tl.text(0.04, 0.05, 'naïve corr. func.',
           transform=ax_tl.transAxes, fontsize=12, va='bottom')

plot_panel(ax_tr, 'log10_wp_corr', 'wp_corr_ref',
           r'$\log_{10}\,w_p$', 'SCOPE corrected')
ax_tr.text(0.04, 0.05, 'weighted corr. func.',
           transform=ax_tr.transAxes, fontsize=12, va='bottom')
ax_tr.legend(
    loc='upper right', ncol=1, fontsize=9,
    bbox_to_anchor=(1.28, 1.0),
)

plot_panel(ax_bl, 'dlog10_wp_naive', 'wp_naive_ref',
           r'$\Delta\log_{10}\,w_p$', '', is_residual=True)

plot_panel(ax_br, 'dlog10_wp_corr', 'wp_corr_ref',
           r'$\Delta\log_{10}\,w_p$', '', is_residual=True)

for ax in axes[1]:
    ax.set_xlabel(r'$\log_{10}\,r_p$  [$h^{-1}$Mpc]')

fig.suptitle(
    f'$w_p$ convergence: naïve vs SCOPE corrected  –  L800/{MODEL}  $z={Z:.2f}$',
    fontsize=14,
)
plt.tight_layout()
plt.show()